# Introduction

This project implements a CNN (Convolutional Neural Network) for the classification of long phrases written by two authors, Kant and Freud (Spanish translation).

Although the classification task was relatively simple, the main objective of this project was to adapt and apply the code presented in the manual to solve similar tasks using locally stored data.

The CNN was implemented using TensorFlow 2.18, although a few modifications were made to adapt the original implementation to both the specific task and TensorFlow 2.21.

The original code was developed by Ganegedara (2022) in the book *Natural Language Processing with TensorFlow 2*.

GitHub repository: https://github.com/thushv89/packt_nlp_tensorflow_2

In [ ]:
# Imports
%matplotlib inline
import numpy as np
import pandas as pd
import os
import random
import tensorflow as tf
import matplotlib
import tensorflow as tf
from sklearn.model_selection import train_test_split
import re
from tensorflow.keras.preprocessing.text import Tokenizer
import tensorflow.keras.backend as K
import tensorflow.keras.layers as layers
import tensorflow.keras.regularizers as regularizers
from tensorflow.keras.models import Model


seed = 54321

%env TF_FORCE_GPU_ALLOW_GROWTH=true

# Data wrangling

## Text cleaning and data splitting

In [ ]:
# Read data
def read_data(data_dir):
    pages_list = []    
    filenames = []
    print("Reading files...")

    i = 0
    for root, dirs, files in os.walk(data_dir):
        for fi, f in enumerate(files):
            if 'readme' in f.lower():
                continue
            
            i += 1
            #print("." * i, f, end='\r')

            file_path = os.path.join(root, f)
            with open(file_path, encoding='utf-8') as text_file:
                lines = [line.strip() for line in text_file]
                page = ' '.join(lines)
                pages_list.append(page)
                filenames.append(file_path)

    print(f"\nDetected {len(pages_list)} text files.")
    return pages_list, filenames

# Path where the two categories folder are
folder_pages = r'PATH\Kant_o_Freud'


data, filenames = read_data(folder_pages)

# FIles counts
print(f"{sum(len(page.split()) for page in data)} total words.")
print("Example (start):", data[0][:50])
print("Example (end):", data[-1][-50:])

In [ ]:
# Paths
base_dir = r'PATH\Kant_o_Freud'
kant_dir = os.path.join(base_dir, "Kant") # Folder name = Kant
freud_dir = os.path.join(base_dir, "Freud") # Folder name = Freud

# Read text files and assign labels
pages = []
labels = []

def read_txts(ruta, etiqueta):
    for filename in sorted(os.listdir(ruta)):
        if filename.lower().endswith(".txt"):
            ruta_completa = os.path.join(ruta, filename)
            with open(ruta_completa, encoding="utf-8") as f:
                text = f.read().strip()
                pages.append(text)
                labels.append(etiqueta)

# Load texts from each author
read_txts(kant_dir, "Kant") # Label = Kant
read_txts(freud_dir, "Freud") # Label = Freud

# Random train-test split
train_pages, test_pages, train_categories, test_categories = train_test_split(
    pages, labels, test_size=0.2, random_state=42
)

# Build dataframes
train_df = pd.DataFrame({'text': train_pages, 'category': train_categories})
test_df = pd.DataFrame({'text': test_pages, 'category': test_categories})

# Display first examples
train_df.head(10)

In [ ]:
# Lists of phrases and words to remove

words_to_replace = {"capítulo", "editorial", "introducción","Página", "freud", "kant"}

phrases_to_replace = ["obras completas", "sigmund freud", "immanuel kant"]

# Function for text cleaning

def clean_text(text, words=[], frases=[]):
    
    # Convert text to lowercase
    text = text.lower()

    # Remove complete phrases
    for frase in frases:
        text = text.replace(frase.lower(), "")

    # Replace special characters
    text = re.sub(r"[«»“”‘’—–…]", " ", text)

    # Normalize common abbreviations
    text = re.sub(r'\bq\b', 'que', text)

    # Remove individual words
    for word in words:
        text = re.sub(rf'\b{re.escape(word)}\b', '', text)

    # Remove numbers and dates
    text = re.sub(r'\b\d+(?:[\.,]\d+)?\b', '', text)

    # Keep only letters and spaces
    text = re.sub(r'[^a-záéíóúüñ\s]', '', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
# Apply text cleaning function
train_df['text'] = train_df['text'].apply(lambda x: clean_text(x, words=words_to_replace, frases=phrases_to_replace))
test_df['text'] = test_df['text'].apply(lambda x: clean_text(x, words=words_to_replace, frases=phrases_to_replace))

In [ ]:
# Shuffle data
train_df = train_df.sample(frac=1.0, random_state=seed) # frac=1.0 means using 100% of the data
test_df = test_df.sample(frac=1.0, random_state=seed)

In [ ]:
# Map categories to IDs
unique_cats = train_df["category"].unique()

# Assign a numeric ID to each category
labels_map = dict(zip(unique_cats, np.arange(unique_cats.shape[0])))

print(f"Label->ID mapping: {labels_map}")

n_classes = len(labels_map)

# Convert string labels into numeric labels
# to avoid issues during model training
train_df["category"] = train_df["category"].map(labels_map)
test_df["category"] = test_df["category"].map(labels_map)

# Display sample data
train_df.head(n=10)

In [ ]:
# Split training and validation data
train_df, valid_df = train_test_split(train_df, test_size=0.1)

print(f"Train size: {train_df.shape}")
print(f"Validation size: {valid_df.shape}")

# Display sample data
train_df.head()

## Tokenization

In [ ]:
# Initialize tokenizer and fit on training data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df["text"].tolist())

# Determine vocabulary size for padding and embedding layers
n_vocab = len(tokenizer.index_word) + 1

print(f"Vocabulary size: {n_vocab}")

Looking for the longest sentence

In [ ]:
# Compute token length distribution and selected percentiles
train_df["text"].str.split(" ").str.len().describe(percentiles=[0.01, 0.5, 0.99])

Padding for setences

In [ ]:
# Convert tokens into sequences of numeric IDs
train_sequences = tokenizer.texts_to_sequences(train_df["text"].tolist())
train_labels = train_df["category"].values

valid_sequences = tokenizer.texts_to_sequences(valid_df["text"].tolist())
valid_labels = valid_df["category"].values

test_sequences = tokenizer.texts_to_sequences(test_df["text"].tolist())
test_labels = test_df["category"].values

# Maximum sequence length based on the 99th percentile
max_seq_length = 619

# Pad shorter sequences and truncate longer ones
preprocessed_train_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    train_sequences, maxlen=max_seq_length, padding='post', truncating='post'
)
preprocessed_valid_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    valid_sequences, maxlen=max_seq_length, padding='post', truncating='post'
)
preprocessed_test_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    test_sequences, maxlen=max_seq_length, padding='post', truncating='post'
)

In [ ]:
K.clear_session()

# Input layer using word IDs
word_id_inputs = layers.Input(shape=(max_seq_length,), dtype='int32')

# Embedding layer with 64-dimensional embeddings
embedding_out = layers.Embedding(input_dim=n_vocab, output_dim=64)(word_id_inputs)


# Convolutional layers with 100 filters
conv1_1 = layers.Conv1D(
    100, kernel_size=3, strides=1, padding='same', activation='relu'
)(embedding_out)
conv1_2 = layers.Conv1D(
    100, kernel_size=4, strides=1, padding='same', activation='relu'
)(embedding_out)
conv1_3 = layers.Conv1D(
    100, kernel_size=5, strides=1, padding='same', activation='relu'
)(embedding_out)

# Concatenate convolution outputs
conv_out = layers.Concatenate(axis=-1)([conv1_1, conv1_2, conv1_3])

# Max pooling
pool_over_time_out = layers.MaxPool1D(pool_size=max_seq_length, padding='valid')(conv_out)

# Flatten output
flatten_out = layers.Flatten()(pool_over_time_out)

# Output layer for classification
out = layers.Dense(
    n_classes, activation='softmax',
    kernel_regularizer=regularizers.l2(0.001) # coef lambda penalization
)(flatten_out)

# Build model
cnn_model = Model(inputs=word_id_inputs, outputs=out)

# Compile model
cnn_model.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)

cnn_model.summary()

# Training

As stated previously, the training process was relatively simple because the main objective of this project was to adapt the original code from the manual to work with locally stored data. Therefore, only five epochs were used.

In [ ]:
# Learning rate reduction callback
lr_reduce_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.1, patience=3, verbose=1,
    mode='auto', min_delta=0.0001, min_lr=0.000001
)

# Train model
# Save training history for potential visualization
history_cnn_model = cnn_model.fit(
    preprocessed_train_sequences, train_labels, 
    validation_data=(preprocessed_valid_sequences, valid_labels),
    batch_size=128, # 3840 samples per epoch
    epochs=5,
    callbacks=[lr_reduce_callback]
)

In [ ]:
# Evaluate model accuracy on the test set
cnn_model.evaluate(preprocessed_test_sequences, test_labels, return_dict=True)

In [ ]:
# Save trained model
cnn_model.save("cnn_model.keras")

# Csv and xslx file with clasifications

In [ ]:
# Load model if the kernel was restarted
cnn_model = tf.keras.models.load_model("cnn_model.keras")

In [ ]:
# Generate predictions on the test set
test_predictions = cnn_model.predict(preprocessed_test_sequences)

## Correct and incorrect predictions

In [ ]:
# Get predicted classes from softmax outputs
predicted_labels = np.argmax(test_predictions, axis=1)

# argmax returns the index of the highest value
# in an array or tensor

# Boolean masks for correct and incorrect predictions
correct_mask = (predicted_labels == test_labels)
incorrect_mask = (predicted_labels != test_labels)

# Count predictions
num_correct = np.sum(correct_mask)
num_incorrect = np.sum(incorrect_mask)

print(f"Correctly classified examples: {num_correct}")
print(f"Incorrectly classified examples: {num_incorrect}")

In [ ]:
# Check label encoding
unique_labels = np.unique(test_labels)
print(unique_labels)

In [ ]:
# Create a dataframe containing classification results and prediction probabilities

def df_classification_results_with_probs(test_texts, test_labels, test_predictions, label_map, n_samples=10):
    pred_labels = np.argmax(test_predictions, axis=1)
    correct_mask = (pred_labels == test_labels)

    rows = []
    n_classes = len(label_map)
    for text, true_label, pred_label, probs, correct in zip(test_texts, test_labels, pred_labels, test_predictions, correct_mask):
        # Format probabilities with 3 decimals, example: {'Freud': 0.923, 'Kant': 0.077}
        prob_dict = {label_map[i]: f"{probs[i]:.3f}" for i in range(n_classes)}
        rows.append({
            "Text": text,
            "True Label": label_map[true_label],
            "Predicted Label": label_map[pred_label],
            "Correct": correct,
            "Probabilities": prob_dict
        })

    df = pd.DataFrame(rows)

    print("Correct examples:")
    display(df[df["Correct"]].head(n_samples))
    
    print("Incorrect examples:")
    display(df[~df["Correct"]].head(n_samples))

    return df # return dataframe for further analysis and saving

In [ ]:
# Display classification results and prediction probabilities

results_df = df_classification_results_with_probs(
    test_texts=test_df["text"].tolist(),
    test_labels=test_df["category"].to_numpy(),
    test_predictions=test_predictions,
    label_map={0: "Freud", 1: "Kant"},
)

# save the results in csv format
results_df.to_csv(r'PATH\df_clasificaciones.csv', sep = ';', encoding= 'latin-1',index=False)
results_df.to_excel(r'PATH\df_clasificaciones.xlsx',index=False)